# LeetCode 3803 — Count Residue Prefixes bằng Python

Link đề: https://leetcode.com/problems/count-residue-prefixes/ · Độ khó: **Easy** · Chủ đề: String, Hash Table, Simulation

Skeleton LeetCode:

```python
class Solution:
    def residuePrefixes(self, s: str) -> int:
        ...
```

> Ghi chú: tên bài, đề, ví dụ, constraints và chữ ký hàm đã được đối chiếu qua WebSearch / leetcode.doocs.org.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một **tiền tố** (prefix) của `s` là "residue" nếu **số ký tự khác nhau** của nó bằng **độ dài của nó chia cho 3 lấy dư**; đếm số tiền tố như vậy.
2. Dùng `set` để đếm ký tự phân biệt, `len`, toán tử `%`, và cắt tiền tố `s[:i]`.
3. Chuyển từ cách "tạo `set` mới cho mỗi tiền tố" sang cách **duy trì tập ký tự khi mở rộng tiền tố** (một lượt duyệt).
4. Viết 5 cách khác nhau (tạo `set` lại mỗi lần, `set` tích lũy, bitmask, mảng đánh dấu, cắt sớm khi quá 2 ký tự phân biệt) và đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: tính cả tiền tố rỗng, quên `% 3`, đếm ký tự phân biệt của **cả chuỗi** thay vì tiền tố, và `range` lệch một đơn vị.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái tiếng Anh thường. Một **tiền tố** của `s` là chuỗi **không rỗng** bắt đầu từ đầu `s` (`s[:1]`, `s[:2]`, ..., `s[:n]`). Một tiền tố được gọi là **residue** nếu:

> **số ký tự phân biệt** của tiền tố **bằng** (**độ dài tiền tố**) `mod 3`.

Trả về **số lượng** tiền tố residue của `s`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"abc"` | `2` | `"a"`: 1 ký tự phân biệt, độ dài 1 mod 3 = 1 ✓; `"ab"`: 2 và 2 mod 3 = 2 ✓; `"abc"`: 3 ký tự phân biệt nhưng 3 mod 3 = 0 ✗ |
| 2 | `"dd"` | `1` | `"d"`: 1 và 1 ✓; `"dd"`: 1 ký tự phân biệt nhưng 2 mod 3 = 2 ✗ |
| 3 | `"bob"` | `2` | `"b"` ✓; `"bo"`: 2 và 2 ✓; `"bob"`: 2 ký tự phân biệt nhưng 3 mod 3 = 0 ✗ |

Constraints: `1 <= s.length <= 100`, `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
for s in ["abc", "dd", "bob"]:
    cnt = 0
    for i in range(1, len(s) + 1):
        p = s[:i]
        ok = len(set(p)) == i % 3
        cnt += ok
        print(f"  s={s!r} tiền tố {p!r}: phân biệt={len(set(p))}, {i} % 3 = {i % 3} -> {'residue' if ok else '-'}")
    print(f"s={s!r} -> {cnt}")

  s='abc' tiền tố 'a': phân biệt=1, 1 % 3 = 1 -> residue
  s='abc' tiền tố 'ab': phân biệt=2, 2 % 3 = 2 -> residue
  s='abc' tiền tố 'abc': phân biệt=3, 3 % 3 = 0 -> -
s='abc' -> 2
  s='dd' tiền tố 'd': phân biệt=1, 1 % 3 = 1 -> residue
  s='dd' tiền tố 'dd': phân biệt=1, 2 % 3 = 2 -> -
s='dd' -> 1
  s='bob' tiền tố 'b': phân biệt=1, 1 % 3 = 1 -> residue
  s='bob' tiền tố 'bo': phân biệt=2, 2 % 3 = 2 -> residue
  s='bob' tiền tố 'bob': phân biệt=2, 3 % 3 = 0 -> -
s='bob' -> 2


## 3. Kiến thức nền

### 3.1. Tiền tố và cắt `s[:i]`

Tiền tố độ dài `i` của `s` là `s[:i]` với `1 <= i <= n`. Nhớ `s[:0]` là chuỗi **rỗng** (không phải tiền tố hợp lệ ở bài này) và `s[:n]` là cả chuỗi.

In [2]:
s = "abc"
print([s[:i] for i in range(1, len(s) + 1)])
print(repr(s[:0]), s[:3] == s)

['a', 'ab', 'abc']
'' True


### 3.2. `set` và số ký tự phân biệt

`set(p)` chỉ giữ các ký tự **khác nhau** của `p`; `len(set(p))` là số ký tự phân biệt. Thêm phần tử đã có vào `set` không làm đổi nó (`set.add`).

In [3]:
p = "bob"
print(set(p), len(set(p)))
seen = set()
for c in p:
    seen.add(c)
    print(c, "->", seen, len(seen))

{'b', 'o'} 2
b -> {'b'} 1
o -> {'b', 'o'} 2
b -> {'b', 'o'} 2


### 3.2b. Phép `%` với độ dài

`i % 3` chỉ nhận giá trị `0`, `1` hoặc `2`. Vì tiền tố **không rỗng** có ít nhất `1` ký tự phân biệt, nên vế phải chỉ có thể khớp khi `i % 3` bằng `1` hoặc `2`; khi `i` chia hết cho `3` thì vế phải bằng `0` — **không bao giờ** khớp.

In [4]:
print([(i, i % 3) for i in range(1, 11)])
print("độ dài chia hết cho 3 luôn thất bại:", [i for i in range(1, 11) if i % 3 == 0])

[(1, 1), (2, 2), (3, 0), (4, 1), (5, 2), (6, 0), (7, 1), (8, 2), (9, 0), (10, 1)]
độ dài chia hết cho 3 luôn thất bại: [3, 6, 9]


### 3.3. Bitmask cho tập chữ cái

Có 26 chữ cái nên một tập chữ cái biểu diễn được bằng một số nguyên 26 bit: bit thứ `c` bằng 1 nếu chữ có thứ tự `c` (`ord(ch) - ord('a')`) xuất hiện. `mask |= 1 << c` thêm một chữ; `mask.bit_count()` (hoặc `bin(mask).count('1')`) đếm số bit 1 = số ký tự phân biệt.

In [5]:
mask = 0
for ch in "bob":
    mask |= 1 << (ord(ch) - ord('a'))
print(bin(mask), bin(mask).count('1'))
print(1 << 0, 1 << 1, 1 << 25)

0b100000000000010 2
1 2 33554432


### 3.4. Mảng đánh dấu 26 ô

Một cách khác để biết "ký tự này đã thấy chưa": `seen = [False] * 26`, ô `ord(ch) - ord('a')` là `True` nếu đã thấy. Đếm `distinct` tăng thêm 1 mỗi lần gặp chữ mới.

In [6]:
seen = [False] * 26
distinct = 0
for ch in "bob":
    idx = ord(ch) - ord('a')
    if not seen[idx]:
        seen[idx] = True
        distinct += 1
    print(ch, idx, "distinct =", distinct)

b 1 distinct = 1
o 14 distinct = 2
b 1 distinct = 2


### 3.5. `break` và thoát sớm

`break` kết thúc vòng lặp ngay. Khi ta chứng minh được "từ điểm này trở đi điều kiện không bao giờ thỏa" thì có thể thoát sớm (xem mục 6).

In [7]:
for i in range(1, 10):
    if i * i > 20:
        print("dừng ở", i)
        break

dừng ở 5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy đi từ trái sang phải, mỗi bước **thêm một ký tự** vào tiền tố và hỏi hai câu:

1. Tiền tố đang dài bao nhiêu? lấy `độ_dài mod 3`.
2. Đang có bao nhiêu ký tự phân biệt?

Nếu hai con số này bằng nhau thì tiền tố đó là residue, cộng 1 vào đáp án.

Tính tay `s = "bob"`:

| i | tiền tố | phân biệt | `i % 3` | residue? |
|---|---|---|---|---|
| 1 | `b` | 1 | 1 | có |
| 2 | `bo` | 2 | 2 | có |
| 3 | `bob` | 2 | 0 | không |

Đáp án `2`. Nhận xét: số ký tự phân biệt **không bao giờ giảm** khi mở rộng tiền tố (chỉ tăng hoặc giữ nguyên) và `i % 3` chỉ nằm trong `{0,1,2}`.

### Bảng tính tay (ví dụ khác)

| s | tiền tố (độ dài) | phân biệt | `i % 3` | residue |
|---|---|---|---|---|
| `"aab"` | `a` (1) | 1 | 1 | có |
| | `aa` (2) | 1 | 2 | không |
| | `aab` (3) | 2 | 0 | không |
| `"abcd"` | `a` | 1 | 1 | có |
| | `ab` | 2 | 2 | có |
| | `abc` | 3 | 0 | không |
| | `abcd` | 4 | 1 | không |

In [8]:
for s in ["aab", "abcd"]:
    seen, res = set(), []
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        res.append((s[:i], len(seen), i % 3, len(seen) == i % 3))
    print(s, res, "->", sum(r[3] for r in res))

aab [('a', 1, 1, True), ('aa', 1, 2, False), ('aab', 2, 0, False)] -> 1
abcd [('a', 1, 1, True), ('ab', 2, 2, True), ('abc', 3, 0, False), ('abcd', 4, 1, False)] -> 2


## 5. Cách 1 — Brute Force / baseline

Với mỗi độ dài `i` từ `1` đến `n`, cắt `p = s[:i]`, tạo `set(p)` mới, so sánh `len(set(p))` với `i % 3`. Dịch thẳng định nghĩa của đề.

Nút thắt: mỗi tiền tố tạo lại `set` từ đầu, tốn `O(i)`; tổng `O(n²)`. Tập ký tự của tiền tố `i + 1` chỉ hơn tiền tố `i` đúng một ký tự, ta lãng phí khi tính lại.

In [9]:
class SolutionBrute:
    def residuePrefixes(self, s):
        count = 0
        for i in range(1, len(s) + 1):
            if len(set(s[:i])) == i % 3:
                count += 1
        return count

print(SolutionBrute().residuePrefixes("abc"), SolutionBrute().residuePrefixes("dd"), SolutionBrute().residuePrefixes("bob"))

2 1 2


**Độ phức tạp:** Time `O(n²)`, Space `O(n)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1:** duy trì **một** `set` tích lũy các ký tự đã thấy; mỗi bước chỉ `add` thêm ký tự mới, `len(seen)` chính là số ký tự phân biệt của tiền tố hiện tại. Tổng thời gian `O(n)`.

**Bước đột phá 2 (thoát sớm):** `i % 3 <= 2` nhưng số ký tự phân biệt không giảm. Nếu số ký tự phân biệt đã `>= 3` thì từ đó về sau nó luôn `>= 3 > i % 3`, nên **không còn tiền tố residue nào** nữa. Có thể `break` ngay.

**Quy tắc "nếu… thì…":**

- Nếu `len(seen) == i % 3` -> cộng 1.
- Nếu `i % 3 == 0` -> không thể residue (vì `len(seen) >= 1`).
- Nếu `len(seen) >= 3` -> dừng sớm, đáp án đã cố định.
- Mọi tiền tố residue có độ dài `i` với `i % 3 ∈ {1, 2}` và 1 hoặc 2 ký tự phân biệt.

**Invariant:** sau bước `i`, `seen` là tập ký tự của `s[:i]`.

In [10]:
# Kiểm chứng: sau khi distinct >= 3 không còn tiền tố residue nào nữa
import random
random.seed(3803)
for _ in range(2000):
    n = random.randint(1, 60)
    s = ''.join(random.choice('abcdef') for _ in range(n))
    seen, flags = set(), []
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        flags.append((len(seen), len(seen) == i % 3))
    first3 = next((t for t, (d, _) in enumerate(flags) if d >= 3), None)
    if first3 is not None:
        assert not any(ok for _, ok in flags[first3:])
print("khi số ký tự phân biệt >= 3, mọi tiền tố sau đó đều không phải residue")

khi số ký tự phân biệt >= 3, mọi tiền tố sau đó đều không phải residue


## 7. Các cách tiếp theo

### Cách 2 — `set` tích lũy (một lượt)

Duyệt `i, ch`; `seen.add(ch)`; so `len(seen)` với `i % 3`. `O(n)` thời gian, `O(1)` bộ nhớ (tối đa 26 phần tử).

In [11]:
class SolutionSet:
    def residuePrefixes(self, s):
        seen = set()
        count = 0
        for i, ch in enumerate(s, 1):
            seen.add(ch)
            if len(seen) == i % 3:
                count += 1
        return count

print(SolutionSet().residuePrefixes("abc"), SolutionSet().residuePrefixes("dd"), SolutionSet().residuePrefixes("bob"))

2 1 2


**Độ phức tạp:** Time `O(n)`, Space `O(1)` (tập tối đa 26 phần tử).

### Cách 3 — Bitmask

Thay `set` bằng một số nguyên: mỗi chữ cái chiếm một bit. `mask |= 1 << (ord(ch) - 97)`; `bin(mask).count('1')` là số ký tự phân biệt. Nhanh và gọn bộ nhớ.

In [12]:
class SolutionMask:
    def residuePrefixes(self, s):
        mask = 0
        count = 0
        for i, ch in enumerate(s, 1):
            mask |= 1 << (ord(ch) - ord('a'))
            if bin(mask).count('1') == i % 3:
                count += 1
        return count

print(SolutionMask().residuePrefixes("abc"), SolutionMask().residuePrefixes("dd"), SolutionMask().residuePrefixes("bob"))

2 1 2


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Mảng đánh dấu và biến đếm `distinct`

Giữ `seen[26]` và `distinct`; chỉ tăng `distinct` khi gặp chữ mới. So sánh `distinct == i % 3` là phép so sánh số nguyên đơn giản, không cần `len(set)`.

In [13]:
class SolutionArray:
    def residuePrefixes(self, s):
        seen = [False] * 26
        distinct = 0
        count = 0
        for i, ch in enumerate(s, 1):
            idx = ord(ch) - ord('a')
            if not seen[idx]:
                seen[idx] = True
                distinct += 1
            if distinct == i % 3:
                count += 1
        return count

print(SolutionArray().residuePrefixes("abc"), SolutionArray().residuePrefixes("dd"), SolutionArray().residuePrefixes("bob"))

2 1 2


**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### Cách 5 — Thoát sớm khi `distinct >= 3`

Dựa trên bước đột phá 2: khi số ký tự phân biệt đạt `3` thì dừng, vì không còn tiền tố residue nào. Hữu ích với chuỗi dài, nhưng bài này `n <= 100` nên chủ yếu để hiểu cấu trúc.

In [14]:
class SolutionEarly:
    def residuePrefixes(self, s):
        seen = set()
        count = 0
        for i, ch in enumerate(s, 1):
            seen.add(ch)
            if len(seen) >= 3:                 # từ đây distinct >= 3 > i % 3: không còn residue
                break
            if len(seen) == i % 3:
                count += 1
        return count

print(SolutionEarly().residuePrefixes("abc"), SolutionEarly().residuePrefixes("dd"), SolutionEarly().residuePrefixes("bob"))

2 1 2


**Độ phức tạp:** Time `O(n)` trong trường hợp xấu (chuỗi chỉ có 1-2 ký tự phân biệt) nhưng thường dừng sau vài bước; Space `O(1)`.

## 8. TRACE từng bước

Trace Cách 2 (`set` tích lũy) với `s = "aabcb"`:

| i | ký tự | `seen` | `len(seen)` | `i % 3` | residue? | `count` |
|---|---|---|---|---|---|---|
| 1 | a | {a} | 1 | 1 | có | 1 |
| 2 | a | {a} | 1 | 2 | không | 1 |
| 3 | b | {a, b} | 2 | 0 | không | 1 |
| 4 | c | {a, b, c} | 3 | 1 | không | 1 |
| 5 | b | {a, b, c} | 3 | 2 | không | 1 |

Đáp án `1`. Từ `i = 4`, `len(seen) = 3` nên đúng như Cách 5 dừng sớm.

In [15]:
def trace_set(s):
    seen, count = set(), 0
    print("s =", s)
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        ok = (len(seen) == i % 3)
        count += ok
        print(f"  i={i} ký tự={ch!r} seen={sorted(seen)} distinct={len(seen)} i%3={i % 3} -> {'residue' if ok else '-'} count={count}")
    return count

assert trace_set("aabcb") == 1
assert trace_set("bob") == 2

s = aabcb
  i=1 ký tự='a' seen=['a'] distinct=1 i%3=1 -> residue count=1
  i=2 ký tự='a' seen=['a'] distinct=1 i%3=2 -> - count=1
  i=3 ký tự='b' seen=['a', 'b'] distinct=2 i%3=0 -> - count=1
  i=4 ký tự='c' seen=['a', 'b', 'c'] distinct=3 i%3=1 -> - count=1
  i=5 ký tự='b' seen=['a', 'b', 'c'] distinct=3 i%3=2 -> - count=1
s = bob
  i=1 ký tự='b' seen=['b'] distinct=1 i%3=1 -> residue count=1
  i=2 ký tự='o' seen=['b', 'o'] distinct=2 i%3=2 -> residue count=2
  i=3 ký tự='b' seen=['b', 'o'] distinct=2 i%3=0 -> - count=2


In [16]:
def trace_early(s):
    seen, count = set(), 0
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        if len(seen) >= 3:
            print(f"  i={i}: distinct={len(seen)} >= 3 -> dừng sớm")
            break
        if len(seen) == i % 3:
            count += 1
            print(f"  i={i}: residue, count={count}")
        else:
            print(f"  i={i}: không")
    return count

assert trace_early("abcxyz") == 2

  i=1: residue, count=1
  i=2: residue, count=2
  i=3: distinct=3 >= 3 -> dừng sớm


## 9. Code nộp LeetCode

In [17]:
from typing import List, Optional

class Solution:
    def residuePrefixes(self, s: str) -> int:
        seen = set()
        count = 0
        for i, ch in enumerate(s, 1):
            seen.add(ch)
            if len(seen) == i % 3:
                count += 1
        return count

## 10. Vì sao thuật toán đúng

- **`seen` đúng là tập ký tự của tiền tố:** ở bước `i`, ta đã thêm đúng các ký tự `s[0..i-1]` vào `seen`, nên `len(seen)` là số ký tự phân biệt của `s[:i]`.
- **Đếm đủ, không thừa:** vòng `enumerate(s, 1)` xét mọi tiền tố không rỗng độ dài `1..n` đúng một lần và không xét tiền tố rỗng.
- **So sánh đúng định nghĩa:** tiền tố là residue khi `distinct == len % 3`; code kiểm tra đúng điều kiện đó với `len = i`.
- **Thoát sớm hợp lệ:** số ký tự phân biệt không giảm khi mở rộng tiền tố; nếu đã `>= 3` thì mọi tiền tố sau đó có `distinct >= 3 > i % 3` (vì `i % 3 <= 2`), nên không có tiền tố residue nào nữa.
- **Độ dài chia hết cho 3:** `i % 3 = 0` còn `distinct >= 1`, nên chúng không bao giờ residue; điều này khớp với ví dụ `"abc"` và `"bob"`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| Tạo `set` lại mỗi tiền tố | `O(n²)` | `O(n)` |
| `set` tích lũy | `O(n)` | `O(1)` |
| Bitmask | `O(n)` | `O(1)` |
| Mảng đánh dấu + `distinct` | `O(n)` | `O(1)` |
| Thoát sớm | `O(n)` xấu nhất, thường nhỏ hơn | `O(1)` |

`n = len(s)`. Bộ nhớ là `O(1)` vì bảng chữ cái chỉ có 26 ký tự. Phải đọc từng ký tự ít nhất cho đến khi có `3` ký tự phân biệt, nên `O(n)` là cận hợp lý cho chuỗi chỉ gồm 1-2 chữ cái.

## 12. Bộ test

Ví dụ của đề, các biên (`n = 1`, một chữ cái lặp lại, hai chữ cái luân phiên, tất cả khác nhau, `n = 100`) và so sánh **tất cả cách** trên input ngẫu nhiên hợp lệ.

In [18]:
impls = [SolutionBrute(), SolutionSet(), SolutionMask(), SolutionArray(), SolutionEarly(), Solution()]

cases = [
    ("abc", 2), ("dd", 1), ("bob", 2), ("a", 1), ("aa", 1), ("aaa", 1), ("aaaa", 2), ("ab", 2), ("abab", 2),
    ("aabcb", 1), ("abcd", 2), ("z" * 100, 34), ("ab" * 50, 34), ("abcdefghijklmnopqrstuvwxyz", 2),
    ("aab", 1), ("ba", 2), ("aba", 2), ("abba", 2),
]
for s, exp in cases:
    assert 1 <= len(s) <= 100 and s.islower() and s.isalpha()
    for impl in impls:
        got = impl.residuePrefixes(s)
        assert got == exp, (type(impl).__name__, s, got, exp)
print("ví dụ + edge OK:", len(cases), "ca")

ví dụ + edge OK: 18 ca


In [19]:
import random, string
random.seed(38030)
for _ in range(1500):
    n = random.randint(1, 100)
    alpha = random.choice(['a', 'ab', 'abc', 'abcd', string.ascii_lowercase])
    s = ''.join(random.choice(alpha) for _ in range(n))
    exp = sum(1 for i in range(1, n + 1) if len(set(s[:i])) == i % 3)
    outs = [impl.residuePrefixes(s) for impl in impls]
    assert all(o == exp for o in outs), (s, outs, exp)
    assert 0 <= exp <= n
print("1500 test ngẫu nhiên (seed=38030): 6 cách giống nhau")

1500 test ngẫu nhiên (seed=38030): 6 cách giống nhau


In [20]:
# Tính chất: đáp án không vượt quá số tiền tố có độ dài không chia hết cho 3; chuỗi 1 chữ cái cho ceil(2n/3)
for _ in range(300):
    n = random.randint(1, 100)
    s = ''.join(random.choice(string.ascii_lowercase) for _ in range(n))
    assert Solution().residuePrefixes(s) <= n - n // 3
for n in range(1, 101):
    assert Solution().residuePrefixes('q' * n) == len([i for i in range(1, n + 1) if i % 3 == 1])
print("tính chất cận trên và chuỗi một chữ cái OK")

tính chất cận trên và chuỗi một chữ cái OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: tính cả tiền tố rỗng

Duyệt `range(0, n)` với `s[:i]` sẽ đưa `s[:0] = ""` vào (có `0` ký tự phân biệt và độ dài `0 % 3 = 0`, nên khớp một cách giả tạo) và bỏ mất cả chuỗi.

In [21]:
def wrong_include_empty(s):
    return sum(1 for i in range(0, len(s)) if len(set(s[:i])) == i % 3)    # SAI: i từ 0, bỏ s[:n]

print(wrong_include_empty("abc"), "!=", Solution().residuePrefixes("abc"))
assert wrong_include_empty("abc") == 3 != 2

3 != 2


### Lỗi 2: quên `% 3`

So sánh `len(set(p))` với **độ dài** `i` thay vì `i % 3` (bỏ mất phép lấy dư).

In [22]:
def wrong_no_mod(s):
    return sum(1 for i in range(1, len(s) + 1) if len(set(s[:i])) == i)    # SAI: thiếu % 3

print(wrong_no_mod("abc"), "!=", Solution().residuePrefixes("abc"))
assert wrong_no_mod("abc") == 3 != 2
assert wrong_no_mod("dd") == 1 == Solution().residuePrefixes("dd")        # trùng hợp ở ca này

3 != 2


### Lỗi 3: đếm ký tự phân biệt của **cả chuỗi**

Điều kiện nói về số ký tự phân biệt **của tiền tố**, không phải của toàn bộ `s`.

In [23]:
def wrong_whole_distinct(s):
    d = len(set(s))                                                           # SAI: của cả chuỗi
    return sum(1 for i in range(1, len(s) + 1) if d == i % 3)

print(wrong_whole_distinct("bob"), "!=", Solution().residuePrefixes("bob"))
assert wrong_whole_distinct("bob") == 1 != 2

1 != 2


### Lỗi 4: `seen` không được cập nhật trước khi so sánh

Nếu so sánh `len(seen)` **trước** khi `add` ký tự hiện tại thì số ký tự phân biệt bị chậm một bước.

In [24]:
def wrong_check_before_add(s):
    seen, count = set(), 0
    for i, ch in enumerate(s, 1):
        if len(seen) == i % 3:                   # SAI: so sánh trước khi add(ch)
            count += 1
        seen.add(ch)
    return count

print(wrong_check_before_add("abc"), "!=", Solution().residuePrefixes("abc"))
assert wrong_check_before_add("abc") == 0 != 2

0 != 2


### Lỗi 5: `break` quá sớm (dừng khi `distinct >= 2`)

Thoát sớm chỉ hợp lệ khi `distinct >= 3`. Dừng ở `>= 2` làm mất tiền tố độ dài 2 như `"ab"`.

In [25]:
def wrong_break_early(s):
    seen, count = set(), 0
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        if len(seen) >= 2:                        # SAI: ngưỡng phải là 3
            break
        if len(seen) == i % 3:
            count += 1
    return count

print(wrong_break_early("abc"), "!=", Solution().residuePrefixes("abc"))
assert wrong_break_early("abc") == 1 != 2

1 != 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Điều kiện trên mọi tiền tố" -> duyệt một lượt và **duy trì trạng thái** (tập ký tự, đếm...) thay vì tính lại từ đầu.
- "Số ký tự phân biệt" -> `set` hoặc bitmask/mảng đánh dấu 26 ô; số này **không giảm** khi mở rộng.
- `i % 3` chỉ có ba giá trị nên mọi điều kiện kiểu "bằng `i % 3`" chỉ có thể thỏa với `distinct <= 2`: có thể thoát sớm.
- Dùng `enumerate(s, 1)` để `i` chính là độ dài tiền tố (tránh lệch một đơn vị).
- Luôn nhớ tiền tố **không rỗng**; chạy thử ví dụ `"abc" -> 2` để bắt lỗi thiếu `% 3` hoặc lệch `range`.
- Biên: `n = 1` luôn cho `1` (`distinct = 1 = 1 % 3`).

## 15. Biến thể của bài

1. **Longest Substring Without Repeating Characters** (LeetCode 3): độ dài đoạn dài nhất không lặp ký tự (cửa sổ trượt).
2. **Number of Distinct Substrings / Count Distinct in Prefix**: số ký tự phân biệt theo từng tiền tố, lưu mảng `D[i]`.
3. **Number of Good Ways to Split a String** (1525): đếm cách chia có số ký tự phân biệt hai nửa bằng nhau (tiền tố + hậu tố).
4. **Replace Elements with Greatest Element on Right Side** (1299): xử lý tiền tố/hậu tố một lượt.
5. **Sum of Beauty of All Substrings** (1781): mở rộng hơn, tần suất trong đoạn.
6. **Residue với modulo `m` bất kỳ:** điều kiện `distinct == i % m`, ngưỡng thoát sớm là `distinct >= m`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| `set` tạo lại | `len(set(s[:i]))` cho mỗi `i` | `O(n²)` | `O(n)` | Oracle kiểm tra |
| `set` tích lũy | `seen.add(ch)` mỗi bước | `O(n)` | `O(1)` | Nộp bài, rõ ràng |
| Bitmask | 26 bit cho 26 chữ cái | `O(n)` | `O(1)` | Cần tốc độ, quen thao tác bit |
| Mảng đánh dấu | `seen[26]` + `distinct` | `O(n)` | `O(1)` | Tránh băm, dễ debug |
| Thoát sớm | `break` khi `distinct >= 3` | `O(n)` | `O(1)` | Chuỗi dài, tối ưu hằng số |

Nộp cách `set` tích lũy.

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().residuePrefixes("aabb")` và `Solution().residuePrefixes("xyzxyz")` ra gì?

**Bài 2 (tính tay):** với `s = "abab"`, lập bảng `(i, tiền tố, distinct, i % 3)` và đếm.

**Bài 3 (điền TODO):** hoàn thành `distinct_per_prefix(s)` trả về list số ký tự phân biệt của từng tiền tố `[d1, d2, ..., dn]`.

**Bài 4 (tự viết lại):** viết `residue_prefix_lengths(s)` trả về **danh sách độ dài** các tiền tố residue (thay vì đếm).

**Bài 5 (mini challenge):** viết `count_residue_mod(s, m)` tổng quát: tiền tố là residue nếu `distinct == len % m`. Kiểm tra `m = 3` khớp `residuePrefixes`.

In [26]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().residuePrefixes("aabb"), Solution().residuePrefixes("xyzxyz"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def distinct_per_prefix_todo(s):
    result = []
    # TODO: duy trì set, mỗi bước thêm ký tự rồi append(len(seen))
    return result

1 2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: với `"aabb"` các tiền tố `a, aa, aab, aabb` có `distinct = 1, 1, 2, 2`; độ dài mod 3 là `1, 2, 0, 1`. Bài 2: `"abab"` chỉ có hai ký tự phân biệt.

**Mức 2 (vừa):** Bài 3: dùng cùng khung `Solution` nhưng `append` thay vì cộng đếm. Bài 4: khi residue thì `append(i)`. Bài 5: thay `3` bằng `m`.

**Mức 3 (gần đáp án):** Bài 1: `"aabb"` -> chỉ `i = 1` thỏa nên `1`; `"xyzxyz"` -> `i=1` và `i=2` thỏa nên `2`. Bài 2: `abab`: `i=1` (1=1) có, `i=2` (2=2) có, `i=3` (2 vs 0) không, `i=4` (2 vs 1) không -> `2`. Bài 4: `residue_prefix_lengths("abc") = [1, 2]`.

## 19. Đáp án tham khảo cho bài tập

In [27]:
# Bài 1, 2
assert Solution().residuePrefixes("aabb") == 1 and Solution().residuePrefixes("xyzxyz") == 2
assert Solution().residuePrefixes("abab") == 2

# Bài 3
def distinct_per_prefix(s):
    seen, result = set(), []
    for ch in s:
        seen.add(ch)
        result.append(len(seen))
    return result
assert distinct_per_prefix("aabb") == [1, 1, 2, 2] and distinct_per_prefix("abc") == [1, 2, 3]

# Bài 4
def residue_prefix_lengths(s):
    seen, out = set(), []
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        if len(seen) == i % 3:
            out.append(i)
    return out
assert residue_prefix_lengths("abc") == [1, 2] and residue_prefix_lengths("dd") == [1]
assert len(residue_prefix_lengths("bob")) == Solution().residuePrefixes("bob")

# Bài 5
def count_residue_mod(s, m):
    seen, count = set(), 0
    for i, ch in enumerate(s, 1):
        seen.add(ch)
        if len(seen) == i % m:
            count += 1
    return count
assert all(count_residue_mod(x, 3) == Solution().residuePrefixes(x) for x in ("abc", "dd", "bob", "aabcb"))
assert count_residue_mod("abcd", 4) == 3                  # i=1,2,3 thỏa (distinct = i), i=4 có 4 % 4 = 0 nên không
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "điều kiện trên mọi tiền tố":**

```python
seen, count = set(), 0
for i, ch in enumerate(s, 1):      # i = độ dài tiền tố
    seen.add(ch)                   # cập nhật trạng thái TRƯỚC khi kiểm tra
    if len(seen) == i % 3:
        count += 1
return count
```

**Checklist trước khi submit:**

- [ ] `i` là độ dài tiền tố (`enumerate(s, 1)`), không bắt đầu từ `0` (không tính tiền tố rỗng)?
- [ ] Đã `add` ký tự hiện tại **trước** khi so sánh?
- [ ] So với `i % 3`, không phải `i`?
- [ ] Đếm ký tự phân biệt của **tiền tố**, không phải cả chuỗi?
- [ ] Nếu thoát sớm: ngưỡng `distinct >= 3`, không phải `>= 2`?
- [ ] Thử `"abc" -> 2`, `"dd" -> 1`, `"bob" -> 2`?
- [ ] Chữ ký đúng `residuePrefixes(self, s: str) -> int`?